In [7]:
import re
from datetime import datetime
from typing import List, Tuple

class Account:
    def __init__(self, owner: str, initial_balance: float = 0.0):
        self.owner = owner
        self._balance = 0.0
        self._transaction_history: List[Tuple[float, str, datetime]] = []
        if initial_balance < 0:
            raise ValueError("Начальный баланс не может быть отрицательным")
        if initial_balance > 0:
            self.deposit(initial_balance)

    @property
    def owner(self) -> str:
        return self._owner

    @owner.setter
    def owner(self, value: str):
        if not isinstance(value, str):
            raise ValueError("Имя владельца должно быть строкой")
        if not re.fullmatch(r'^[A-Za-zА-Яа-яЁё]+\s+[A-Za-zА-Яа-яЁё]+$', value):
            raise ValueError("Имя владельца должно быть в формате 'Имя Фамилия' кириллицей или латиницей")
        if not all(part[0].isupper() for part in value.split()):
            raise ValueError("Имя и фамилия должны начинаться с заглавной буквы")
        self._owner = value

    @property
    def balance(self) -> float:
        return self._balance

    def deposit(self, amount: float):
        if amount < 0:
            raise ValueError("Сумма пополнения не может быть отрицательной")
        self._balance += amount
        self._transaction_history.append((amount, 'deposit', datetime.now()))

    def withdraw(self, amount: float):
        if amount < 0:
            raise ValueError("Сумма снятия не может быть отрицательной")
        if amount > self._balance:
            raise ValueError("Недостаточно средств на счёте")
        self._balance -= amount
        self._transaction_history.append((-amount, 'withdraw', datetime.now()))

    def get_last_large_transactions(self, n: int) -> List[Tuple[float, str, datetime]]:
        if n <= 0:
            return []
        sorted_by_amount = sorted(
            self._transaction_history,
            key=lambda x: abs(x[0]),
            reverse=True
        )
        return sorted_by_amount[:n]


class CheckingAccount(Account):
    account_type = "checking"


class SavingsAccount(Account):
    account_type = "savings"

    def apply_interest(self, rate: float):
        if rate < 0:
            raise ValueError("Процентная ставка не может быть отрицательной")
        interest = self._balance * (rate / 100)
        self._balance += interest
        self._transaction_history.append((interest, 'interest', datetime.now()))

    def withdraw(self, amount: float):
        if amount < 0:
            raise ValueError("Сумма снятия не может быть отрицательной")
        if amount > self._balance:
            raise ValueError("Недостаточно средств на счёте")
        if amount > self._balance * 0.5:
            raise ValueError("Нельзя снять более 50% от текущего баланса")
        self._balance -= amount
        self._transaction_history.append((-amount, 'withdraw', datetime.now()))

In [8]:
acc1 = CheckingAccount("Иван Петров", 1000)
acc2 = SavingsAccount("Anna Smith", 2000)

acc1.deposit(500)
acc1.withdraw(200)

acc2.deposit(300)
acc2.apply_interest(7)
acc2.withdraw(500)

print(acc2.get_last_large_transactions(3))

[(2000, 'deposit', datetime.datetime(2025, 11, 5, 15, 59, 19, 135449)), (-500, 'withdraw', datetime.datetime(2025, 11, 5, 15, 59, 19, 135542)), (300, 'deposit', datetime.datetime(2025, 11, 5, 15, 59, 19, 135510))]
